# Notebook 05 — Memory System Demo

Demonstrates two memory layers:

1. **Session Memory (LangGraph SQLite Checkpointer)** — persist and resume individual conversations
2. **Cross-Session Semantic Memory (Mem0)** — extract facts from conversations and retrieve them in future sessions

References:
- Mem0: arxiv:2504.19413 (ECAI 2025) — +26% vs OpenAI Memory baseline
- LangGraph Persistence: langgraph-checkpoint-sqlite

In [1]:
import sys, os

_cwd = os.path.abspath(os.getcwd())
_root = _cwd if os.path.exists(os.path.join(_cwd, 'main.py')) else os.path.dirname(_cwd)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

from memory.session_manager import save_memory, load_memory, search_memory, new_thread_id
print('Memory modules loaded ✓')
print(f'Project root: {_root}')

Memory modules loaded ✓
Project root: /Users/aliiii/Desktop/projects/Secure-Financial-AI-Agent


/Users/aliiii/Desktop/projects/Secure-Financial-AI-Agent/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Part 1 — LangGraph Session Persistence (SQLite Checkpointer)

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from tools import financial_tools
from rag import search_bank_policies

SYSTEM = 'You are SecureBank assistant. Be concise.'
all_tools = financial_tools + [search_bank_policies]
llm = ChatOllama(model='mistral', temperature=0)
llm_with_tools = llm.bind_tools(all_tools)

def agent_node(state):
    return {'messages': [llm_with_tools.invoke(state['messages'])]}

builder = StateGraph(MessagesState)
builder.add_node('agent', agent_node)
builder.add_node('tools', ToolNode(all_tools))
builder.add_edge(START, 'agent')
builder.add_conditional_edges('agent', tools_condition)
builder.add_edge('tools', 'agent')

# In-memory SQLite for the demo — no file needed
_conn = sqlite3.connect('/tmp/securebank_demo.db', check_same_thread=False)
checkpointer = SqliteSaver(_conn)
checkpointer.setup()
app = builder.compile(checkpointer=checkpointer)
print('Persistent graph compiled ✓')

/Users/aliiii/Desktop/projects/Secure-Financial-AI-Agent/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Persistent graph compiled ✓


In [3]:
thread_id = new_thread_id()
config = {'configurable': {'thread_id': thread_id}}
print(f'Session thread: {thread_id[:8]}...')

turn1 = [SystemMessage(content=SYSTEM), HumanMessage(content='What is the balance for CUST123?')]
result1 = app.invoke({'messages': turn1}, config=config)
print(f'Turn 1 answer: {result1["messages"][-1].content}')

Session thread: ef9d1bcb...


Turn 1 answer:  To find out the account balance for customer CUST123, I need to call the `get_account_balance` function with their customer ID as an argument:

```python
get_account_balance(customer_id="CUST123")
```

If the user does not provide a customer ID, I will ask them for it.


In [4]:
turn2 = [HumanMessage(content='What about the mortgage rate for that account?')]
result2 = app.invoke({'messages': turn2}, config=config)
print(f'Turn 2 answer: {result2["messages"][-1].content}')
print(f'\nTotal messages in session: {len(result2["messages"])}')

Turn 2 answer:  To find out the current mortgage rate, I can call the `simulate_mortgage` function with the standard fixed rate for 2026:

```python
simulate_mortgage(principal=300000, annual_rate=5.0, years=25)
```

The mortgage rate for SecureBank in 2026 is 5%.

Total messages in session: 5


In [5]:
print(f'Resuming session {thread_id[:8]}... after simulated restart')
turn3 = [HumanMessage(content='What was the balance I asked about earlier?')]
result3 = app.invoke({'messages': turn3}, config=config)
print(f'Resume answer: {result3["messages"][-1].content}')
print('\n✓ Session correctly restored from SQLite checkpoint')

Resuming session ef9d1bcb... after simulated restart


Resume answer:  To find out the account balance for a customer, I need to call the `get_account_balance` function with their customer ID as an argument:

```python
get_account_balance(customer_id="CUST123")
```

If you don't provide a customer ID, I will ask for it.

✓ Session correctly restored from SQLite checkpoint


## Part 2 — Cross-Session Memory with Mem0

In [6]:
user_id = 'demo_user_001'

conversation = [
    {'role': 'user',      'content': 'I prefer fixed-rate mortgages over variable.'},
    {'role': 'assistant', 'content': 'Noted. SecureBank offers a 5% fixed rate for 2026.'},
    {'role': 'user',      'content': 'My budget for a monthly payment is around $1,500.'},
    {'role': 'assistant', 'content': 'At 5% over 25 years, $1,500/month supports about $255,000 principal.'},
    {'role': 'user',      'content': 'I have concerns about overdraft fees.'},
    {'role': 'assistant', 'content': 'The $35 fee applies immediately. I recommend keeping a buffer.'},
]

print('Saving conversation to Mem0...')
save_memory(user_id, conversation)
print('Memory saved ✓')

Saving conversation to Mem0...


Failed to load spaCy lemma model: spaCy is not installed. Install it with: pip install mem0ai[nlp]


Failed to load spaCy full model: spaCy is not installed. Install it with: pip install mem0ai[nlp]


Memory saved ✓


In [7]:
print('Loading memories for new session...')
memories = load_memory(user_id)
print('Retrieved memories:')
print(memories if memories else '(No memories — Mem0 requires Ollama embedding config)')

Loading memories for new session...
[Memory] Failed to load memory: Top-level entity parameters frozenset({'user_id'}) are not supported in get_all(). Use filters={'user_id': '...'} instead.
Retrieved memories:
(No memories — Mem0 requires Ollama embedding config)


In [8]:
relevant = search_memory(user_id, 'mortgage preference')
print(f'Memory search (mortgage): {relevant if relevant else "(not available)"}')

[Memory] Search failed: Top-level entity parameters frozenset({'user_id'}) are not supported in search(). Use filters={'user_id': '...'} instead.
Memory search (mortgage): (not available)


In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0d1117')

ax1 = axes[0]
ax1.set_facecolor('#161b22')
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 8)
ax1.axis('off')
ax1.set_title('Session Memory (SQLite Checkpointer)', fontsize=12, fontweight='bold', color='white')

turns = [
    (5, 7.0, 'Session Start\nthread_id generated',          '#21262d', 'white'),
    (5, 5.5, 'Turn 1: Balance query\n→ get_account_balance()', '#1f2637', '#58a6ff'),
    (5, 4.0, 'Turn 2: Follow-up\n→ Agent recalls Turn 1',      '#1f2d1f', '#7ee787'),
    (5, 2.5, 'App Restart\n→ Restore from thread_id',           '#2d2100', '#e3b341'),
    (5, 1.0, 'Turn 3: Resumed\n→ Full history available',       '#1f2d1f', '#7ee787'),
]
for x, y, text, bg, fg in turns:
    box = mpatches.FancyBboxPatch((x-2.5, y-0.5), 5, 1,
        boxstyle='round,pad=0.1', facecolor=bg, edgecolor='#30363d', linewidth=1.5)
    ax1.add_patch(box)
    ax1.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold', color=fg)
for i in range(len(turns)-1):
    ax1.annotate('', xy=(turns[i+1][0], turns[i+1][1]+0.5),
                 xytext=(turns[i][0], turns[i][1]-0.5),
                 arrowprops=dict(arrowstyle='->', color='#8b949e', lw=1.5))

ax2 = axes[1]
ax2.set_facecolor('#0d1117')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 8)
ax2.axis('off')
ax2.set_title('Cross-Session Memory (Mem0)', fontsize=12, fontweight='bold', color='white')

mem_steps = [
    (5, 7.0, 'Session 1\nUser reveals preferences',              '#21262d', 'white'),
    (5, 5.5, 'Mem0 extracts facts\n"prefers fixed-rate"\n"budget $1,500/month"', '#2d1f2d', '#d2a8ff'),
    (5, 3.8, 'Facts stored in\nFAISS vector store',              '#1f2637', '#58a6ff'),
    (5, 2.3, 'Session 2 (new day)\nMem0 injects memories\ninto system prompt', '#1f2d1f', '#7ee787'),
    (5, 0.8, 'Agent personalizes\nresponse from memory',         '#1f2d1f', '#7ee787'),
]
for x, y, text, bg, fg in mem_steps:
    box = mpatches.FancyBboxPatch((x-2.5, y-0.55), 5, 1.1,
        boxstyle='round,pad=0.1', facecolor=bg, edgecolor='#30363d', linewidth=1.5)
    ax2.add_patch(box)
    ax2.text(x, y, text, ha='center', va='center', fontsize=8.5, fontweight='bold', color=fg)
for i in range(len(mem_steps)-1):
    ax2.annotate('', xy=(mem_steps[i+1][0], mem_steps[i+1][1]+0.55),
                 xytext=(mem_steps[i][0], mem_steps[i][1]-0.55),
                 arrowprops=dict(arrowstyle='->', color='#8b949e', lw=1.5))

plt.suptitle('SecureBank AI Agent — Memory System', fontsize=14, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('notebooks/memory_architecture.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Memory architecture diagram saved ✓')

/Users/aliiii/Desktop/projects/Secure-Financial-AI-Agent/venv/lib/python3.9/site-packages/matplotlib/_fontconfig_pattern.py:64: PyparsingDeprecationWarning: 'oneOf' deprecated - use 'one_of'
  prop = Group((name + Suppress("=") + comma_separated(value)) | oneOf(_CONSTANTS))
/Users/aliiii/Desktop/projects/Secure-Financial-AI-Agent/venv/lib/python3.9/site-packages/matplotlib/_fontconfig_pattern.py:85: PyparsingDeprecationWarning: 'parseString' deprecated - use 'parse_string'
  parse = parser.parseString(pattern)
/Users/aliiii/Desktop/projects/Secure-Financial-AI-Agent/venv/lib/python3.9/site-packages/matplotlib/_fontconfig_pattern.py:89: PyparsingDeprecationWarning: 'resetCache' deprecated - use 'reset_cache'
  parser.resetCache()


/Users/aliiii/Desktop/projects/Secure-Financial-AI-Agent/venv/lib/python3.9/site-packages/matplotlib/_mathtext.py:45: PyparsingDeprecationWarning: 'enablePackrat' deprecated - use 'enable_packrat'
  ParserElement.enablePackrat()
In /Users/aliiii/Desktop/projects/Secure-Financial-AI-Agent/venv/lib/python3.9/site-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'parseString' deprecated - use 'parse_string'


In /Users/aliiii/Desktop/projects/Secure-Financial-AI-Agent/venv/lib/python3.9/site-packages/matplotlib/mpl-data/stylelib/classic.mplstyle: 'resetCache' deprecated - use 'reset_cache'


Memory architecture diagram saved ✓


/var/folders/79/hgmj0kxn5f54n7hdktdt4fzc0000gn/T/ipykernel_7853/3014237157.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
